In [23]:
from ArrayGen import *
import numpy as np
import matplotlib.pyplot as plt
from functools import partial
from scipy.optimize import differential_evolution, minimize
from datetime import datetime
import os
import pandas as pd

start = datetime.now()

In [24]:
# Original Project
# a = 0.1
# er = 2.55
# h=0.001524
a = 0.08
er = 9.8
h = 0.003175
center_freq = 1575.42e6

# Parameters from the Original Project
# param = [0.58574961, 0.5814837,  1.66819249, 1.64960034]
param = [0.35925613, 0.35375391, 1.61708165, 1.60821528]


In [ ]:
### User parameters
dtr = np.pi/180

# Positions of the patches
Alphas = np.array([11.25, -11.25, 33.75, -33.75, 56.25, -56.25, 78.75, -78.75])
Betas = np.array([0, 0, 0, 0, 0, 0, 0, 0])
Gammas = np.array([0, 0, 0, 0, 0, 0, 0, 0])

# Initial Guess
# mods = [1.8, 2.15]
# phases = [80*dtr, 0*dtr]
# mods = [1, 1]
# phases = [0, 0]

# Mask
ang_range = 176
Dref = [0]*(ang_range+1)

lim1 = 1
lim2 = 1

w1 = 0.65
w2 = 0.35

len_mods = int(len(Alphas)/2 - 1)
bounds = [(0, 5)]*len_mods + [(0, 2*np.pi)]*len_mods
if len(Betas) != len(Alphas) or len(Gammas) != len(Alphas) or len(Dref) != int(ang_range+1):
    print("Incompatible length!")
    exit()

In [26]:
positions = [Alphas, Betas, Gammas]

keyDir = "case_a=" + str(a*1000) + "mm_er=" + str(er) + "_h=" + str(h*1000) + "mm_f=" + str(int(center_freq/1e6)) + "MHz"

folder_name = "Results/" + str(len(Alphas)) + "_Patches_at_" + str(np.array(positions).tolist())
cur_folder = os.getcwd()
results_folder = os.path.join(cur_folder, folder_name)
os.makedirs(results_folder, exist_ok=True)

with open(os.path.join(results_folder,"history.txt"), "w", encoding="utf-8") as f:
    f.write(f"Started at {start.strftime('%d/%m/%Y %H:%M:%S')}\n")

array = partial(array_synth, positions=positions, keyDir=keyDir)

ig = 0

def cost_function(estim):
    mods, phases = np.split(estim, 2)
    if not (np.all((phases >= 0) & (phases <= 2*np.pi)) and np.all((mods >= 0) & (mods <= 5))):
        return 1e10
    amptds = np.r_[1, 1, np.repeat(mods * np.exp(1j * phases), 2)]
    amptds /= np.linalg.norm(amptds)

    # Function
    angles, Erhcp_dB, Elhcp_dB, Erhcp_full_dB, Elhcp_full_dB, AR_th90, AR_ph90 = array(amptds)

    # Range
    idx_center = 270

    # Theta = 90°
    Er = Elhcp_dB[int(idx_center-ang_range/2):int(idx_center+ang_range/2+1)]
    abs_diff = np.abs(Er-Dref)
    ARs = AR_th90[int(idx_center-ang_range/2):int(idx_center+ang_range/2+1)]

    delta1 = (abs_diff > lim1).astype(int)
    delta2 = (ARs > lim2).astype(int)
    
    cost = (w1*np.sum(delta1*(abs_diff-lim1)**2) + w2*np.sum(delta2*(ARs-lim2)**2))/(ang_range+1)

    global ig
    ig = ig + 1
    print('    i = ', ig, '; ', np.round(cost, 6), ', for: ', mods, phases/dtr, "\n")
    return np.round(cost, 6)

In [ ]:
res_global = differential_evolution(cost_function, bounds, maxiter=4, popsize=3)

estimates = res_global.x

mods0, phases0 = np.split(estimates, 2)
print('Estimates Differential Evolution: ', mods0, phases0/dtr)

    i =  1 ;  31.591926 , for:  [0.84689718 0.85893289 4.37450737] [123.362327   134.04744268 356.52802927] 

    i =  2 ;  6.974612 , for:  [4.39864283 3.89045525 2.7945306 ] [239.00197999 243.18231958 256.7077888 ] 

    i =  3 ;  90.049138 , for:  [2.2076966 2.2756871 3.6351277] [ 23.75825652 311.90616331  95.11374564] 

    i =  4 ;  29.176943 , for:  [0.75464307 0.36074828 4.83568867] [317.10394923 169.90997364  61.58466911] 

    i =  5 ;  4.171728 , for:  [3.4121009  1.90203397 1.5488408 ] [267.24786686 269.83537166 285.62518488] 

    i =  6 ;  54.119378 , for:  [0.12995104 1.27952835 2.34538988] [254.72715738  93.99277156 221.73065275] 

    i =  7 ;  42.553733 , for:  [1.84995675 3.74228566 4.0708953 ] [321.73643173 102.96453732 306.21623356] 

    i =  8 ;  30.350172 , for:  [1.62184369 2.82383042 2.57757237] [299.9244242  333.60966404 169.82878934] 

    i =  9 ;  16.522615 , for:  [3.26006692 4.75799804 3.21071527] [343.17060182   1.48425256  15.95621793] 

    i =  10 ;  

In [ ]:
ig = 0
result = minimize(cost_function, estimates, method='nelder-mead', tol=1e-10, options={"fatol": 1e-3, "xatol": 1e-2})
estimates = result.x

mods, phases = np.split(estimates, 2)
amptds = np.r_[1, 1, np.repeat(mods * np.exp(1j * phases), 2)]
amptds /= np.linalg.norm(amptds)

end = datetime.now()

    i =  1 ;  0.412133 , for:  [1.58757444 0.55956272 0.84438277] [295.61948872 284.05943675 275.64012289] 

    i =  2 ;  0.379214 , for:  [1.66695317 0.55956272 0.84438277] [295.61948872 284.05943675 275.64012289] 

    i =  3 ;  0.406777 , for:  [1.58757444 0.58754086 0.84438277] [295.61948872 284.05943675 275.64012289] 

    i =  4 ;  0.440785 , for:  [1.58757444 0.55956272 0.88660191] [295.61948872 284.05943675 275.64012289] 

    i =  5 ;  0.441376 , for:  [1.58757444 0.55956272 0.84438277] [310.40046316 284.05943675 275.64012289] 

    i =  6 ;  0.395902 , for:  [1.58757444 0.55956272 0.84438277] [295.61948872 298.26240859 275.64012289] 

    i =  7 ;  0.52941 , for:  [1.58757444 0.55956272 0.84438277] [295.61948872 284.05943675 289.42212903] 

    i =  8 ;  0.32457 , for:  [1.61403402 0.56888877 0.85845582] [300.5464802  288.7937607  261.85811674] 

    i =  9 ;  0.250336 , for:  [1.6272638  0.57355179 0.86549234] [303.00997594 291.16092267 248.0761106 ] 

    i =  10 ;  0.3382

In [ ]:
print('Estimates Nelder-Mead: ', mods, phases/dtr)
print("Time:", end-start)
with open(results_folder+"/history.txt", "a", encoding="utf-8") as f:
    f.write("\nMinimum cost: " + str(cost_function(estimates)) + "\n")
    f.write('\nEstimates for Differential Evolution: ' + str(mods0) + "; " + str(phases0/dtr) + "\n")
    f.write('\nEstimated ratios: ' + str(mods) + "\nEstimated phases (deg): " + str(phases/dtr) + "\n")
    f.write('\n\nFor each Patch: ' + str(amptds) + "\n")
    f.write("\nFinished at " + end.strftime('%d/%m/%Y %H:%M:%S')+ "\n Duration: " + str(end-start))

In [ ]:
angles, Erhcp_dB, Elhcp_dB, Erhcp_full_dB, Elhcp_full_dB, AR_th90, AR_ph90 = array(amptds)

tick_positions = [-150, -120, -90, -60, -30, 0, 30, 60, 90, 120, 150, 180]

tick_labels = [
    "-150°", "-120°", "-90°", "-60°", "-30°",
    "0°", "30°", "60°", "90°", "120°", "150°", "180°"
]

fig, axs = plt.subplots(3, 2, figsize=(14, 12))

plots = [
    (Erhcp_dB, r"$E_{RHCP}$ (dB) ($\theta=90$°)"),
    (Elhcp_dB, r"$E_{LHCP}$ (dB) ($\theta=90$°)"),
    (Erhcp_full_dB, r"$E_{RHCP}$ (dB) ($\varphi=90$°)"),
    (Elhcp_full_dB, r"$E_{LHCP}$ (dB) ($\varphi=90$°)"),
    (AR_th90, r"Axial Ratio (dB) ($\theta=90$°)"),
    (AR_ph90, r"Axial Ratio (dB) ($\varphi=90$°)"),
]

lines = []

for ax, (data, title) in zip(axs.flatten(), plots):

    line = ax.plot(angles, data, marker="o", markersize=3)[0]

    lines.append(line)

    ax.set_title(title)
    ax.set_xlabel("Angle (deg)")
    ax.set_ylabel("dB")
    if title in [r"$E_{LHCP}$ (dB) ($\theta=90$°)"]:
        ax.axhline(Dref[0]-lim1, color = "green", linestyle = "--")
        ax.axvline(90-ang_range/2, color = "green", linestyle = "--")
        ax.axvline(90+ang_range/2, color = "green", linestyle = "--")
    if title in [r"Axial Ratio (dB) ($\theta=90$°)"]:
        ax.axhline(lim2, color = "green", linestyle = "--")
        ax.axvline(90-ang_range/2, color = "green", linestyle = "--")
        ax.axvline(90+ang_range/2, color = "green", linestyle = "--")
    ax.set_xlim(-180, 180)
    ax.grid(True)

    if "full" in title.lower():
        ax.set_xticks(tick_positions)
        ax.set_xticklabels(tick_labels)

plt.tight_layout()
plt.savefig(os.path.join(results_folder, "Optmized_Fields_AR.png"))
plt.show(block=True)

In [ ]:
df = pd.DataFrame({
    "Angle [deg]": angles,
    "E_RHCP (Theta=90) [dB]": Erhcp_dB,
    "E_LHCP (Theta=90) [dB]": Elhcp_dB,
    "AR (Theta=90) [dB]": AR_th90,
    "E_RHCP (Phi=90) [dB]": Erhcp_full_dB,
    "E_LHCP (Phi=90) [dB]": Elhcp_full_dB,
    "AR (Phi=90) [dB]": AR_ph90
})

df.to_csv(os.path.join(results_folder, "Fields_Data.csv"), index=False, encoding="utf-8")

# Workbench

In [ ]:
import tempfile
from ansys.aedt.core import Hfss
from ansys.aedt.core.modeler.cad.polylines import PolylineSegment
import time 
import matplotlib.image as mpimg

start = time.time()
AEDT_VERSION = "2025.2"
NUM_CORES = 4
NG_MODE = False
ConvP = 2

temp_folder = tempfile.TemporaryDirectory(suffix=".ansys")

project_name = os.path.join(temp_folder.name, "Spherical_Array.aedt")
design_name = "Sphere"
hfss = Hfss(version=AEDT_VERSION,
            non_graphical=NG_MODE,
            project=project_name,
            design = design_name,
            new_desktop=True,
            solution_type="Modal",
            )

In [ ]:
materials = {2.55 : "Arlon AD255A (tm)", # er = 2.55
4.5 : "Rogers TMM 4 (tm)", # er = 4.5
9.8 : "Rogers TMM 10i (tm)" # er = 9.8
}

# material = "Arlon AD255A (tm)" # er = 2.55
# material = "Rogers TMM 4 (tm)" # er = 4.5
# material = "Rogers TMM 10i (tm)" # er = 9.8
# h = 0.003175

# er = float(hfss.materials[material].permittivity.value)

material = materials[er]
band = np.round(center_freq*er/50e6,0)*1e6
freq_range = [center_freq-band/2, center_freq+band/2]

geom = [a, er, h]

In [ ]:
hfss["a"] = str(1000*a)+"mm"
hfss["numseg"] = "19"
hfss["h"] = str(1000*h)+"mm"

hfss["Dthetaa"] = str(round(param[0]/dtr,4)) + "deg"
hfss["Dphia"] = str(round(param[1]/dtr,4)) + "deg"
hfss["thetap1"] = str(round(param[2]/dtr,4)) + "deg"
hfss["phip1"] = str(round(param[3]/dtr,4)) + "deg"
if len(param) == 5:
    hfss["alphac"] = str(round(param[4]/dtr,2)) + "deg"
else:
    hfss["alphac"] = "0deg"

hfss["Rteflon"] = "2.05mm"
Rteflon= 0.00205
hfss["rprobe"] = "0.65mm"
rprobe = 0.00065
hfss["Hprobe"] = "15mm"
Hprobe = 0.015
orange = [255,128,64]
red = [255,0,0]
blue = [0,255,255]

In [ ]:
hfss.modeler.set_working_coordinate_system("Global")

start_point = ["0mm", "0mm", "a"]
center_point = ["0mm", "0mm", "0mm"]

Gnd_Sphere_Name = "Gnd_Sphere"
Gnd_Sphere = hfss.modeler.create_polyline(
    points=[start_point, start_point],
    segment_type=[PolylineSegment(
        segment_type="AngularArc",
        arc_center=center_point,
        arc_angle="-180deg",
        arc_plane="ZX",
        num_seg="numseg"
    ), PolylineSegment(
        segment_type="Line"
    )],
    name=Gnd_Sphere_Name
)

hfss.modeler.cover_lines(Gnd_Sphere_Name)

hfss.modeler.sweep_around_axis(Gnd_Sphere_Name, "360deg", number_of_segments="2*numseg")

Gnd_Sphere.material_name = "copper"
Gnd_Sphere.color = orange

start_point = ["0mm", "0mm", "a+h"]

Diel_Sphere_Name = "Diel_Sphere"
Diel_Sphere = hfss.modeler.create_polyline(
    points=[start_point, start_point],
    segment_type=[PolylineSegment(
        segment_type="AngularArc",
        arc_center=center_point,
        arc_angle="-180deg",
        arc_plane="ZX",
        num_seg="numseg"
    ), PolylineSegment(
        segment_type="Line"
    )],
    name=Diel_Sphere_Name
)

hfss.modeler.cover_lines(Diel_Sphere_Name)

hfss.modeler.sweep_around_axis(Diel_Sphere_Name, "360deg", number_of_segments="2*numseg")

Diel_Sphere.material_name =  material

hfss.modeler.subtract(Diel_Sphere_Name, Gnd_Sphere_Name, keep_originals=True)

hfss.modeler.fit_all()

Probe_Sphere_Name = "Patch_Sphere"
Probe_Sphere = hfss.modeler.create_polyline(
    points=[["0mm", "0mm", "a+3*h"], ["0mm", "0mm", "a+3*h"]],
    segment_type=[PolylineSegment(
        segment_type="AngularArc",
        arc_center=center_point,
        arc_angle="-180deg",
        arc_plane="ZX",
        num_seg="numseg"
    ), PolylineSegment(
        segment_type="Line"
    )],
    name=Probe_Sphere_Name
)

hfss.modeler.cover_lines(Probe_Sphere_Name)

hfss.modeler.sweep_around_axis(Probe_Sphere_Name, "360deg", number_of_segments="2*numseg")

hfss.modeler.subtract(Probe_Sphere_Name, [Gnd_Sphere_Name, Diel_Sphere_Name], keep_originals=True)

material_tef = "Teflon (tm)"

for i in range(1,len(Alphas)+1):
    Patch_Name = "Patch" + str(i)
    Patch = hfss.modeler.create_polyline(
        points=start_point,
        segment_type=PolylineSegment(
            segment_type="AngularArc",
            arc_center=center_point,
            arc_angle="-180deg",
            arc_plane="ZX",
            num_seg="numseg"
        ),
        name=Patch_Name
    )

    hfss.modeler.sweep_around_axis(Patch_Name, "360deg", number_of_segments="2*numseg")
    
    hfss["Alpha"+str(i)] = str(Alphas[i-1])+"deg"
    hfss["Beta"+str(i)] = str(Betas[i-1])+"deg"
    hfss["Gamma"+str(i)] = str(Gammas[i-1])+"deg"

    coord = hfss.modeler.create_coordinate_system(mode = "zxz", origin=center_point, reference_cs="Global",name=Patch_Name+"_CS", phi = "Alpha" + str(i), theta = "Beta"+str(i))

    Cylinder_Name = "Cylinder"+str(i)
    Cylinder = hfss.modeler.create_polyline(
        points=[["0mm","0mm","a+2*h"],["0mm","a+2*h","a+2*h"],["0mm","a+2*h","-a-2*h"],["0mm","0mm","-a-2*h"],["0mm","0mm","a+2*h"]],
        segment_type=[PolylineSegment(
            segment_type="Line",
        ), PolylineSegment(
            segment_type="Line"
        ), PolylineSegment(
            segment_type="Line"
        ), PolylineSegment(
            segment_type="Line"
        )],
        name=Cylinder_Name
    )
    hfss.modeler.cover_lines(Cylinder_Name)
    hfss.modeler.rotate(Cylinder, "Z", angle="-Dphia/2")
    hfss.modeler.sweep_around_axis(Cylinder_Name, axis="Z", sweep_angle="Dphia-360deg")
    Cylinder.rotate("Y", "Gamma"+str(i))

    Cone_Upper_Name = "Cone_Up_" + str(i)
    Cone_Upper = hfss.modeler.create_cone(orientation = "Z", origin = ["0mm","0mm","a+2*h"],bottom_radius = "(a+2*h)*tan(90deg-Dthetaa/2)", top_radius=0, height = "-(a+2*h)", name = Cone_Upper_Name)
    Cone_Upper.rotate("Y", "Gamma"+str(i))

    Cone_Lower_Name = "Cone_Down_" + str(i)
    Cone_Lower = hfss.modeler.create_cone(orientation = "Z", origin = ["0mm","0mm","-(a+2*h)"],bottom_radius = "(a+2*h)*tan(90deg-Dthetaa/2)", top_radius=0, height = "a+2*h", name = Cone_Lower_Name)
    Cone_Lower.rotate("Y", "Gamma"+str(i))

    hfss["alphaa"] = "acos(-sin(PI/2-Dthetaa/2)^2*cos(PI-Dphia)-cos(PI/2-Dthetaa/2)^2)"
    hfss["alphay"] = "atan(1/(tan(PI/2-Dthetaa/2)*cos(PI/2-Dphia/2)))"

    CylinderTrunc_Name = "CylinderTrunc"+str(i)
    CylinderTrunc = hfss.modeler.create_polyline(
        points=[["0mm","0mm","a+2*h"],["0mm","a+2*h","a+2*h"],["0mm","a+2*h","-a-2*h"],["0mm","0mm","-a-2*h"],["0mm","0mm","a+2*h"]],
        segment_type=[PolylineSegment(
            segment_type="Line",
        ), PolylineSegment(
            segment_type="Line"
        ), PolylineSegment(
            segment_type="Line"
        ), PolylineSegment(
            segment_type="Line"
        )],
        name=CylinderTrunc_Name
    )
    hfss.modeler.cover_lines(CylinderTrunc_Name)
    hfss.modeler.rotate(CylinderTrunc, "Z", angle="-alphaa/2+alphac/2")
    hfss.modeler.sweep_around_axis(CylinderTrunc_Name, axis="Z", sweep_angle="alphaa-alphac-360deg")
    CylinderTrunc.rotate("Y", "alphay")

    hfss.modeler.subtract(Patch_Name, [Cylinder_Name, Cone_Lower_Name, Cone_Upper_Name, CylinderTrunc_Name], keep_originals=False)
    hfss.assign_finite_conductivity(Patch_Name)

    Probe_Name = "Probe"+str(i)
    Probe = hfss.modeler.create_cylinder(orientation="Z", origin=["0mm","0mm","a+h"], radius="rprobe", height="-Hprobe", name = Probe_Name, material="copper")

    Teflon_Name = "Teflon"+str(i)
    Teflon = hfss.modeler.create_cylinder(orientation="Z", origin=["0mm","0mm","a+h/5"], radius="Rteflon", height="4*h/5-Hprobe", name = Teflon_Name, material=material_tef)

    hfss.modeler.subtract(Teflon_Name, Probe_Name, keep_originals=True)

    hfss.wave_port(assignment=hfss.modeler[Teflon_Name].faces[0], name = "Wave_Port"+str(i), deembed="Hprobe-h")

    hfss.modeler.rotate([Probe_Name, Teflon_Name], "Y", angle="thetap1")
    hfss.modeler.rotate([Probe_Name, Teflon_Name], "Z", angle="phip1")
    hfss.modeler.rotate([Probe_Name, Teflon_Name], "Y", angle="Gamma"+str(i))
    hfss.modeler.subtract(Gnd_Sphere_Name,[Teflon_Name, Probe_Name], keep_originals=True)
    hfss.modeler.subtract(Teflon_Name, Diel_Sphere_Name, keep_originals=True)
    hfss.modeler.subtract(Diel_Sphere_Name, Probe_Name, keep_originals=True)
    hfss.modeler.subtract(Probe_Name, Probe_Sphere_Name, keep_originals=True)

    hfss.modeler.set_working_coordinate_system("Global")

    hfss.modeler[Patch_Name].color = orange

hfss.modeler.delete(Probe_Sphere_Name)
for obj in hfss.modeler.object_names:
    hfss.modeler[obj].transparency = 0

hfss.materials[material_tef].permittivity.value = "1.9"
hfss.materials[material_tef].update()

In [ ]:
source_powers = np.round(np.abs(amptds),2)
source_phases = np.round(np.angle(amptds, deg=True), 0)
sources = {}
for i in range(len(Alphas)):
    sources["Wave_Port"+str(i+1)+":1"] = (str(source_powers[i])+"W", str(source_phases[i])+"deg")
hfss.edit_sources(sources, include_port_post_processing=False)

open_box = hfss.create_open_region(frequency=str(center_freq/1e9)+"GHz", boundary = "PML")

setup = hfss.create_setup(name="Resonance_Band", Frequency = str(center_freq/1e9)+"GHz", MaximumPasses=20, MinimumPasses=2, MinimumConvergedPasses=ConvP, PercentRefinement=30)

hfss.post.update_report_dynamically = True

exp_cache = setup.enable_expression_cache(report_type="Far Fields", expressions="dB(AxialRatioValue)" ,intrinsics="Theta=\'90deg\' Phi=\'90deg\'", isrelativeconvergence= False, conv_criteria='0.3')
converge = hfss.post.available_report_quantities(report_category="Terminal Solution Data",quantities_category="Expression Converge")
hfss.post.create_report([converge[0], converge[1]], setup_sweep_name="Resonance_Band : AdaptivePass", variations=hfss.get_nominal_variation(),primary_sweep_variable="Pass",report_category="Terminal Solution Data",plot_type="Rectangular Plot", plot_name="Convergence")

disc_sweep = setup.add_sweep(name="Sweep_Discrete", sweep_type="Discrete", RangeStart=str(freq_range[0]/1e9)+"GHz", RangeEnd=str(freq_range[1]/1e9)+"GHz", SaveFields=True)
setup.props['MinimumConvergedPasses'] = 5
disc_sweep.props['RangeCount'] = 31
disc_sweep.update()
sweep = [freq_range[0], freq_range[1], 31]
freqs = np.linspace(freq_range[0], freq_range[1], 31)

setup.update()

In [ ]:
setup.analyze()

In [ ]:
reportZ = hfss.post.create_report(expressions=["re(ActiveZ(Wave_Port1:1))","im(ActiveZ(Wave_Port1:1))"], variations=hfss.get_nominal_variation(), setup_sweep_name="Sweep_Discrete", plot_name="Z_11_HFSS_Final")
hfss.post.export_report_to_csv(project_dir=results_folder,plot_name="Z_11_HFSS_Final")

Rad_3D = hfss.field_setups[0]
print(Rad_3D.name)
Rad_3D.theta_step = "1deg"
Rad_3D.phi_step = "1deg"
Rad_3D.theta_start = "-180deg"

variations = hfss.get_nominal_variation()
variations["Freq"] = ["All"]
variations["Theta"] = ["90deg"]
variations["Phi"] = ["90deg"]
axialratio_ffd_plot = hfss.post.create_report(expressions="dB(AxialRatioValue)",
                                             setup_sweep_name="Sweep_Discrete",
                                             variations=variations,
                                             primary_sweep_variable="Freq",
                                             context="3D",
                                             report_category="Far Fields",
                                             plot_type="Rectangular Plot",
                                             plot_name="Axial Ratio (dB)"
                                            )
hfss.post.export_report_to_csv(project_dir=results_folder,plot_name="Axial Ratio (dB)")

variations = hfss.get_nominal_variation()
variations["Freq"] = ["All"]
variations["Theta"] = ["90deg"]
variations["Phi"] = ["90deg"]
axialratio_ffd_plot = hfss.post.create_report(expressions=["dB(GainLHCP)", "dB(GainRHCP)"],
                                             setup_sweep_name="Sweep_Discrete",
                                             variations=variations,
                                             primary_sweep_variable="Freq",
                                             context="3D",
                                             report_category="Far Fields",
                                             plot_type="Rectangular Plot",
                                             plot_name="Gain (dB)"
                                            )
hfss.post.export_report_to_csv(project_dir=results_folder,plot_name="Gain (dB)")

reportS = hfss.post.create_report(expressions=["dB(S(Wave_Port1,Wave_Port1))"], setup_sweep_name="Sweep_Discrete", plot_name="S_11", variations = hfss.get_nominal_variation())
hfss.post.export_report_to_csv(project_dir=results_folder,plot_name="S_11")

selections=["Diel_Sphere"]
for i in range(len(Alphas)):
    selections.append("Patch"+str(i+1))

hfss.post.export_model_picture(full_name=results_folder.replace('/', '\\')+"\\Array_View.png", show_axis=True, show_grid=True, show_ruler=True, show_region='Default', selections=selections, field_selections="all", orientation='back', width=0, height=0)

img = mpimg.imread(results_folder+"/Array_View.png")  
plt.imshow(img)
plt.axis("off")
plt.show()

In [ ]:
mesh_plot = hfss.post.create_fieldplot_volume(["Diel_Sphere"], quantity="Mesh", field_type="Fields", setup="Resonance_Band : LastAdaptive", plot_name="Diel_Mesh")

hfss.post.export_model_picture(full_name=results_folder.replace('/', '\\')+"\\Back_View.png", show_axis=True, show_grid=True, show_ruler=True, show_region='Default', selections=["Diel_Sphere", "Patch1"], field_selections="all", orientation='back', width=0, height=0)

img = mpimg.imread(results_folder+"/Back_View.png")  
plt.imshow(img)
plt.axis("off")
plt.show()

In [ ]:
hfss.post.update_report_dynamically = True

variations = hfss.get_nominal_variation()
variations["Freq"] = [str(center_freq/1e9)+"GHz"]
variations["Theta"] = ["90deg"]
variations["Phi"] = ["All"]
axialratio_ffd_plot = hfss.post.create_report(expressions="dB(AxialRatioValue)",
                                             setup_sweep_name="Sweep_Discrete",
                                             variations=variations,
                                             primary_sweep_variable="Phi",
                                             context="3D",
                                             report_category="Far Fields",
                                             plot_type="Rectangular Plot",
                                             plot_name="Axial Ratio Theta90 (dB)"
                                            )
hfss.post.export_report_to_csv(project_dir=results_folder,plot_name="Axial Ratio Theta90 (dB)")

variations = hfss.get_nominal_variation()
variations["Freq"] = [str(center_freq/1e9)+"GHz"]
variations["Theta"] = ["All"]
variations["Phi"] = ["90deg"]
axialratio_ffd_plot = hfss.post.create_report(expressions="dB(AxialRatioValue)",
                                             setup_sweep_name="Sweep_Discrete",
                                             variations=variations,
                                             primary_sweep_variable="Theta",
                                             context="3D",
                                             report_category="Far Fields",
                                             plot_type="Rectangular Plot",
                                             plot_name="Axial Ratio Phi90 (dB)"
                                            )
hfss.post.export_report_to_csv(project_dir=results_folder,plot_name="Axial Ratio Phi90 (dB)")

variations = hfss.get_nominal_variation()
variations["Freq"] = [str(center_freq/1e9)+"GHz"]
variations["Theta"] = ["All"]
variations["Phi"] = ["All"]
gain_ffd_plot = hfss.post.create_report(expressions=["dB(GainLHCP)", "dB(GainRHCP)"],
                                             setup_sweep_name="Sweep_Discrete",
                                             variations=variations,
                                             context="3D",
                                             report_category="Far Fields",
                                             plot_type="3D Polar Plot",
                                             plot_name="Gain 3D (dB)"
                                            )
hfss.post.export_report_to_csv(project_dir=results_folder,plot_name="Gain 3D (dB)")

variations = hfss.get_nominal_variation()
variations["Freq"] = [str(center_freq/1e9)+"GHz"]
variations["Theta"] = ["90deg"]
variations["Phi"] = ["All"]
gain_ffd_plot = hfss.post.create_report(expressions=["dB(GainLHCP)", "dB(GainRHCP)"],
                                             setup_sweep_name="Sweep_Discrete",
                                             variations=variations,
                                             primary_sweep_variable="Phi",
                                             context="3D",
                                             report_category="Far Fields",
                                             plot_type="Radiation Pattern",
                                             plot_name="Gain Theta90 (dB)"
                                            )
hfss.post.export_report_to_csv(project_dir=results_folder,plot_name="Gain Theta90 (dB)")

variations = hfss.get_nominal_variation()
variations["Freq"] = [str(center_freq/1e9)+"GHz"]
variations["Theta"] = ["All"]
variations["Phi"] = ["90deg"]
gain_ffd_plot = hfss.post.create_report(expressions=["dB(GainLHCP)", "dB(GainRHCP)"],
                                             setup_sweep_name="Sweep_Discrete",
                                             variations=variations,
                                             primary_sweep_variable="Theta",
                                             context="3D",
                                             report_category="Far Fields",
                                             plot_type="Radiation Pattern",
                                             plot_name="Gain Phi90 (dB)"
                                            )
hfss.post.export_report_to_csv(project_dir=results_folder,plot_name="Gain Phi90 (dB)")

reportS = hfss.post.create_report(expressions=["dB(S(Wave_Port1,Wave_Port1))"], setup_sweep_name="Sweep_Discrete", plot_name="S_11", variations = hfss.get_nominal_variation())
hfss.post.export_report_to_csv(project_dir=results_folder,plot_name="S_11")

In [ ]:
variations = hfss.available_variations.nominal_values
variations["Freq"] = [str(center_freq/1e9)+"GHz"]
variations["Theta"] = ["All"]
variations["Phi"] = ["All"]
gain_ffd_plot = hfss.post.create_report(expressions=["re(rEPhi)"],
                                             setup_sweep_name="Sweep_Discrete",
                                             variations=variations,
                                             context="3D",
                                             report_category="Far Fields",
                                             plot_type="3D Polar Plot",
                                             plot_name="rE_Phi_re"
                                            )
hfss.post.export_report_to_csv(project_dir=results_folder,plot_name="rE_Phi_re")

gain_ffd_plot = hfss.post.create_report(expressions=["im(rEPhi)"],
                                             setup_sweep_name="Sweep_Discrete",
                                             variations=variations,
                                             context="3D",
                                             report_category="Far Fields",
                                             plot_type="3D Polar Plot",
                                             plot_name="rE_Phi_im"
                                            )
hfss.post.export_report_to_csv(project_dir=results_folder,plot_name="rE_Phi_im")

gain_ffd_plot = hfss.post.create_report(expressions=["re(rETheta)"],
                                             setup_sweep_name="Sweep_Discrete",
                                             variations=variations,
                                             context="3D",
                                             report_category="Far Fields",
                                             plot_type="3D Polar Plot",
                                             plot_name="rE_Theta_re"
                                            )
hfss.post.export_report_to_csv(project_dir=results_folder,plot_name="rE_Theta_re")

gain_ffd_plot = hfss.post.create_report(expressions=["im(rETheta)"],
                                             setup_sweep_name="Sweep_Discrete",
                                             variations=variations,
                                             context="3D",
                                             report_category="Far Fields",
                                             plot_type="3D Polar Plot",
                                             plot_name="rE_Theta_im"
                                            )
hfss.post.export_report_to_csv(project_dir=results_folder,plot_name="rE_Theta_im")

In [ ]:
reportEff3 = hfss.post.reports_by_category.antenna_parameters("RadiationEfficiency", setup="Resonance_Band : Sweep_Discrete", infinite_sphere="3D")
reportEff3.report_type = "Data Table"
reportEff3.create()
reportEff3.plot_name = "Efficiency"
reportEff3.variations = hfss.get_nominal_variation()
data = reportEff3.get_solution_data()
data.export_data_to_csv(results_folder+"\\Efficiency_Data.csv")

In [ ]:
with open(results_folder+"/history.txt", "a", encoding="utf-8") as f:
    f.write("\n3dB Beamwidth Results:\n")
    def calc_3db_angle(gg, angulos, label):
        N = len(gg)
        p = np.argmax(gg)
        t = gg[p] - 3

        r = p
        while gg[(r+1)%N] > t:
            r = (r+1)%N
        ar = angulos[(r+1)%N]

        l = p
        while gg[(l-1)%N] > t:
            l = (l-1)%N
        al = angulos[(l-1)%N]

        bw = (ar - al) % 360
        f.write("\n3dB Beamwidth " + label + ": "+ str(bw)+ " degrees")

    # RHCP e LHCP
    angulos = np.arange(-180,181,1)
    angles = np.arange(0, 360, 30)
    angles_labels = ['0°', '30°', '60°', '90°', '120°', '150°', '180°', '-150°', '-120°', '-90°', '-60°', '-30°']
    dtr = np.pi/180

    df_RHCP_th90 = pd.read_csv(results_folder + '/Gain Theta90 (dB).csv')
    df_LHCP_th90 = pd.read_csv(results_folder + '/Gain Theta90 (dB).csv')

    Erhcp = df_RHCP_th90["dB(GainRHCP) []"].to_numpy(dtype=float)
    Elhcp = df_LHCP_th90["dB(GainLHCP) []"].to_numpy(dtype=float)

    gain = np.max([np.max(Erhcp), np.max(Elhcp)])

    fig, axs2 = plt.subplots(2, 2, figsize=(8, 8), subplot_kw={'projection': 'polar'})

    gg = Erhcp
    axs2[0, 0].plot(angulos * dtr, gg, color='red')
    axs2[0, 0].plot(angulos * dtr, [gain-3]*len(angulos), color='b', linestyle='--')
    axs2[0, 0].set_title('Amplitude '+r'$E_{RHCP}$'+' (dB)')
    axs2[0, 0].set_xlabel('Angle Phi (degrees) for Theta = 90°')
    axs2[0, 0].set_theta_zero_location('N')
    axs2[0, 0].set_theta_direction(-1)
    axs2[0, 0].set_rlim(gain-30,gain)
    axs2[0, 0].set_thetagrids(angles)
    axs2[0, 0].set_rlabel_position(45)

    calc_3db_angle(gg, angulos, 'RHCP Theta=90°')

    gg = Elhcp
    axs2[0, 1].plot(angulos * dtr, gg, color='red')
    axs2[0, 1].plot(angulos * dtr, [gain-3]*len(angulos), color='b', linestyle='--')
    axs2[0, 1].set_title('Amplitude '+r'$E_{LHCP}$'+' (dB)')
    axs2[0, 1].set_xlabel('Angle Phi (degrees) for Theta = 90°')
    axs2[0, 1].set_theta_zero_location('N')
    axs2[0, 1].set_theta_direction(-1)
    axs2[0, 1].set_rlim(gain-30,gain)
    axs2[0, 1].set_thetagrids(angles)
    axs2[0, 1].set_rlabel_position(45)

    calc_3db_angle(gg, angulos, 'LHCP Theta=90°')

    df_RHCP_ph90 = pd.read_csv(results_folder + '/Gain Phi90 (dB).csv')
    df_LHCP_ph90 = pd.read_csv(results_folder + '/Gain Phi90 (dB).csv')

    Erhcp_full = df_RHCP_ph90["dB(GainRHCP) []"].to_numpy(dtype=float)
    Elhcp_full = df_LHCP_ph90["dB(GainLHCP) []"].to_numpy(dtype=float)

    gg = Erhcp_full
    axs2[1, 0].plot(angulos * dtr, gg, color='red')
    axs2[1, 0].plot(angulos * dtr, [gain-3]*len(angulos), color='b', linestyle='--')
    axs2[1, 0].set_title('Amplitude '+r'$E_{RHCP}$'+' (dB)')
    axs2[1, 0].set_xlabel('Angle Theta (degrees) for Phi = 90°')
    axs2[1, 0].set_theta_zero_location('N')
    axs2[1, 0].set_theta_direction(-1)
    axs2[1, 0].set_rlim(gain-30,gain)
    axs2[1, 0].set_thetagrids(angles, labels=angles_labels)
    axs2[1, 0].set_rlabel_position(45)

    calc_3db_angle(gg, angulos, 'RHCP Phi=90°')

    gg = Elhcp_full
    axs2[1, 1].plot(angulos * dtr, gg, color='red')
    axs2[1, 1].plot(angulos * dtr, [gain-3]*len(angulos), color='b', linestyle='--')
    axs2[1, 1].set_title('Amplitude '+r'$E_{LHCP}$'+' (dB)')
    axs2[1, 1].set_xlabel('Angle Theta (degrees) for Phi = 90°')
    axs2[1, 1].set_theta_zero_location('N')
    axs2[1, 1].set_theta_direction(-1)
    axs2[1, 1].set_rlim(gain-30,gain)
    axs2[1, 1].set_thetagrids(angles, labels=angles_labels)
    axs2[1, 1].set_rlabel_position(45)

    calc_3db_angle(gg, angulos, 'LHCP Phi=90°')

    plt.tight_layout()
    plt.savefig(results_folder+'/RHCP_LHCP_Patterns.png', format='png', dpi=300)

    end = time.time()
    f.write("\nSimulation Finished at: " + datetime.now().strftime('%d/%m/%Y %H:%M:%S') + "\n")
    f.write("------------------------------------------------------")

# Workbench

In [ ]:
hfss.save_project()
hfss.release_desktop()
temp_folder.cleanup()